# Домашнее задание: Глубокое обучение на реальных данных

В этом задании вы будете работать с датасетом Oxford-IIIT Pet — набором изображений 37 пород кошек и собак. Ваша задача: пройти полный цикл разработки модели классификации, применить все изученные методы регуляризации и проанализировать результаты.

**Цели:**
1. Загрузить и визуализировать реальные данные.
2. Построить baseline-модель и диагностировать переобучение.
3. Применить регуляризацию (L2, Dropout, BatchNorm, аугментацию).
4. Сравнить кривые обучения и сделать выводы.

Начнём с установки необходимых библиотек.

Импортируйте все необходимые библиотеки: PyTorch, torchvision, matplotlib, numpy, tqdm.
Также определите устройство (GPU, если доступно).

Теперь мы используем датасет **Oxford-IIIT Pet** — 37 пород кошек и собак. Датасет доступен в `torchvision.datasets.OxfordIIITPet`.

1. Создайте трансформацию для **обучения** (только ToTensor и нормализация).
2. Создайте трансформацию для **теста** (аналогично).
3. Загрузите тренировочный набор с `split='trainval'` и тестовый с `split='test'`.
4. Выведите количество классов и примеров.

Для нормализации используем стандартные для ImageNet: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`.

Посмотрите, что за картинки мы будем классифицировать.

Напишите функцию, которая выводит сетку 3x3 случайных изображений из тренировочного набора с подписями классов. Используйте `matplotlib`. Не забудьте денормализовать картинки для правильного отображения.

Создайте загрузчики данных:
- `train_loader` с `batch_size=32` и `shuffle=True`.
- `test_loader` с `batch_size=32` и `shuffle=False`.

Мы будем использовать маленький размер батча, чтобы модель быстрее обновлялась и мы могли наблюдать динамику.

Создайте класс `SimpleCNN` со следующей архитектурой:
- Свёрточный слой: 3 канала → 32 канала, kernel=3, padding=1
- ReLU, MaxPool(2,2)
- Свёрточный слой: 32 → 64, kernel=3, padding=1
- ReLU, MaxPool(2,2)
- Свёрточный слой: 64 → 128, kernel=3, padding=1
- ReLU, MaxPool(2,2)
- Глобальный AvgPool (можно AdaptiveAvgPool2d(1))
- Полносвязный слой: 128 → 37 (количество классов)

Не забудьте, что после свёрток нужно выпрямить тензор перед полносвязным слоем.

Напишите функцию `train_model`, которая:
- Принимает модель, загрузчики, число эпох, learning rate, weight decay (для L2), и флаг `use_clipping`.
- Использует `CrossEntropyLoss` и оптимизатор Adam.
- Для каждой эпохи вычисляет train loss, train accuracy, test loss, test accuracy.
- Если `use_clipping=True`, применяет gradient clipping с `max_norm=1.0`.
- Возвращает словарь с историей метрик.

Функция должна выводить прогресс по эпохам (можно использовать tqdm).

Обучите модель `SimpleCNN` без регуляризации (weight_decay=0, use_clipping=False) на 20 эпохах.

Сохраните историю в переменную `history_no_reg`.

Постройте два графика для эксперимента без регуляризации:
1. Train и Test Loss по эпохам.
2. Train и Test Accuracy по эпохам.

Есть ли признаки переобучения? На какой эпохе test loss перестаёт снижаться или начинает расти? Сделайте вывод.

Создайте модифицированную модель `RegularizedCNN`, которая добавляет:
- Dropout (p=0.5) после каждого свёрточного блока (после пула).
- Возможность включать Batch Normalisation после свёрточных слоёв (перед ReLU).

Добавьте параметры в `__init__`: `use_dropout=True, dropout_prob=0.5, use_batchnorm=True`. Классов по-прежнему 37.

Обучите модель `RegularizedCNN` с параметрами:
- `use_dropout=True, use_batchnorm=True`
- `weight_decay=1e-4` (в оптимизаторе)
- `use_clipping=False` (или True, поэкспериментируйте)

Сохраните историю в `history_reg`.

Постройте на одном графике:
- Test loss для модели без регуляризации и с регуляризацией.
- Test accuracy для обеих моделей.

Сделайте вывод: насколько регуляризация помогла? Уменьшился ли разрыв между train и test?

Теперь добавим аугментацию. Создайте новый набор трансформаций для тренировки:
- RandomResizedCrop(224)
- RandomHorizontalFlip(p=0.5)
- ColorJitter(brightness=0.2, contrast=0.2)
- ToTensor
- Нормализация

Загрузите Oxford-IIIT Pet заново с этой трансформацией (можно использовать тот же корень, данные уже скачаны).
Создайте новый загрузчик `train_loader_aug`.

Обучите модель `RegularizedCNN` на аугментированных данных с теми же гиперпараметрами (weight_decay=1e-4).
Сохраните историю в `history_aug`.

**Важно:** на тесте аугментация не применяется, используйте тот же `test_loader`, что и раньше.

Постройте финальный график, на котором будут Test Accuracy для трёх экспериментов:
1. Baseline без регуляризации.
2. С регуляризацией (L2+Dropout+BN).
3. С регуляризацией + аугментация.

Какой эксперимент показал лучший результат? Почему?

Напишите небольшой вывод (5-10 предложений) о том:
- Какие методы регуляризации оказались наиболее эффективными на этом датасете.
- Насколько аугментация улучшила результат.
- Что бы вы попробовали ещё для улучшения качества (например, изменение архитектуры, подбор гиперпараметров, предобученные модели).

Это задание без кода, просто текст.